# Renters (specialty-ltv) v1.5.0 - loss ratio refit validation (cat excluded)Renters schema differs from classic: `projected_ntr_N_lr` carries the appliedloss ratio directly, so the propagation check is a direct comparison.

In [ ]:
%env ENV_FOR_DYNACONF = prod%env DYNACONF_GIT_BRANCH = feature/B-2895893%env DYNACONF_GIT_CHECKOUT = feature/B-2895893

In [ ]:
import numpy as npimport pandas as pdimport ltv_helpers.non_spark_helpers as nsfrom specialty_ltv import paths as ppd.options.display.max_rows = 2000pd.options.display.max_columns = 500

In [ ]:
P_SCORED_NEW = p.score_internal_resultsP_SCORED_PRIOR = ("tmx-smsiweb/specialty-ltv/prod/"                  "<PRIOR_RELEASE>/ltv_calc/score_internal_results/")  # >>> fill inprint(P_SCORED_NEW)

## I. Fits

In [ ]:
ntr_gt0_fits = [16, 88, 90]   # 71 is not one of them - no separate nt0spl_fits  = {71: [-0.043571, 0.625204]}prod_fits = {71: [-0.05045,  0.7259]}LINE_NAMES = {71: "Renters"}MAX_NT6 = 18def eval_fit(nt6, line, fits):    f = fits[line]    if nt6 < 2 and line in ntr_gt0_fits:        return f[2]    return f[1] + f[0] * (min(nt6, MAX_NT6) / 2)

### Fit delta

In [ ]:
pd.DataFrame([    {"nt6": n, "ntr": min(n, MAX_NT6) / 2,     "prod": eval_fit(n, 71, prod_fits),     "new": eval_fit(n, 71, spl_fits)}    for n in range(10)]).assign(diff=lambda d: d["new"] - d["prod"]).round(4)

## II. Load

In [ ]:
COLS = ["drv_line", "ply_pt_state_cd", "ply_ntr_nbr",        "lifetime_premium", "lifetime_loss", "cat_loss_amt",        "balance_amt", "bal_factor", "lr_cat",        "loss_ratio_x_cat_yr1_target", "loss_ratio_x_cat_yr2_target",        "loss_ratio_x_cat_yr3_target",        "premium_new", "premium_renew"] + [f"projected_ntr_{n}_lr" for n in range(10)]df = ns.read_parquet_s3_to_pandas(P_SCORED_NEW)[COLS]df["drv_line"] = df["drv_line"].astype(int)df.shape

## III. Propagation check`projected_ntr_N_lr` is the applied loss ratio, so this compares it to the fitdirectly. The column name says `ntr` but classic indexes the equivalent arrayby nt6 (ntr = N/2), so both conventions are tested - whichever gives ~0 gap isthe right one.

In [ ]:
rows = []for n in range(10):    obs = df[f"projected_ntr_{n}_lr"]    rows.append({        "n": n,        "obs_mean": obs.mean(),        "obs_nunique": obs.nunique(),        "as_nt6": eval_fit(n, 71, spl_fits),          # ntr = n/2        "as_ntr": eval_fit(2 * n, 71, spl_fits),      # ntr = n    })r = pd.DataFrame(rows)r["gap_nt6"] = (r["obs_mean"] - r["as_nt6"]).abs()r["gap_ntr"] = (r["obs_mean"] - r["as_ntr"]).abs()r.round(6)

In [ ]:
# Once the convention is settled, tighten to a per-policy max gap.# Set IDX = "nt6" or "ntr" from the table above.IDX = "nt6"   # >>> set from the result abovechk = []for n in range(10):    exp = eval_fit(n if IDX == "nt6" else 2 * n, 71, spl_fits)    gap = (df[f"projected_ntr_{n}_lr"] - exp).abs()    chk.append({"n": n, "expected": exp, "max_gap": gap.max(), "mean_gap": gap.mean()})chk = pd.DataFrame(chk)chk["pass"] = chk["max_gap"] < 1e-4chk.round(6)

In [ ]:
# Same against prod_fits - should FAIL if the new fits landed.pd.DataFrame([    {"n": n,     "expected_prod": eval_fit(n if IDX == "nt6" else 2 * n, 71, prod_fits),     "max_gap_vs_prod": (df[f"projected_ntr_{n}_lr"]                         - eval_fit(n if IDX == "nt6" else 2 * n, 71, prod_fits)).abs().max()}    for n in [0, 2, 4, 9]]).round(6)

## IV. Cat components`lr_cat`, `cat_loss_amt` and `loss_ratio_x_cat_yr1/2/3_target` exist, so cat iscarried separately and the ex-cat LR has its own balance targets - same shape asclassic. Check whether the refit belongs in whatever config feeds those targets.

In [ ]:
df[["lr_cat", "loss_ratio_x_cat_yr1_target", "loss_ratio_x_cat_yr2_target",    "loss_ratio_x_cat_yr3_target", "bal_factor"]].mean().round(4)

## V. v1.5.0 vs prior releaseNot a clean cat-impact read - the two releases score different books. Watch`d_premium`. `balance_amt` is the discriminating column: if it moves to cancelthe cat change, the balance step absorbed it and nothing reaches the P&L.

In [ ]:
def lr_summary(path):    d = ns.read_parquet_s3_to_pandas(path)[        ["drv_line", "lifetime_premium", "lifetime_loss",         "cat_loss_amt", "balance_amt"]]    d["drv_line"] = d["drv_line"].astype(int)    g = d.groupby("drv_line", as_index=False).agg(        premium=("lifetime_premium", "sum"), loss=("lifetime_loss", "sum"),        cat=("cat_loss_amt", "sum"), bal=("balance_amt", "sum"))    g["lr_ex_cat"] = g["loss"] / g["premium"]    g["lr_total"] = (g["loss"] + g["cat"]) / g["premium"]    g["cat_share"] = g["cat"] / (g["loss"] + g["cat"])    return gcmp = lr_summary(P_SCORED_NEW).merge(    lr_summary(P_SCORED_PRIOR), on="drv_line", suffixes=("_new", "_old"))for c in ["lr_ex_cat", "lr_total", "premium", "cat", "bal"]:    cmp[f"d_{c}"] = cmp[f"{c}_new"] - cmp[f"{c}_old"]cmp["prem_pct"] = cmp["d_premium"] / cmp["premium_old"]cmp[["drv_line", "lr_ex_cat_old", "lr_ex_cat_new", "d_lr_ex_cat",     "lr_total_old", "lr_total_new", "d_lr_total",     "d_cat", "d_bal", "d_premium", "prem_pct"]].round(4)

### Did the cat component change?`lr_total - lr_ex_cat` is the applied cat load. Unchanged across releases while`lr_ex_cat` dropped means the old fitted LR carried cat *and* the cat load wasadded on top - the refit removed a double count rather than creating a gap.This is what classic showed. Confirm it holds for renters.

In [ ]:
cmp["cat_comp_old"] = cmp["lr_total_old"] - cmp["lr_ex_cat_old"]cmp["cat_comp_new"] = cmp["lr_total_new"] - cmp["lr_ex_cat_new"]cmp["d_cat_comp"] = cmp["cat_comp_new"] - cmp["cat_comp_old"]cmp["implied"] = cmp["lr_ex_cat_old"] * (1 - cmp["cat_share_new"])cmp[["drv_line", "cat_comp_old", "cat_comp_new", "d_cat_comp",     "cat_share_new", "lr_ex_cat_old", "implied", "lr_ex_cat_new"]].round(4)

## Open items

In [ ]:
# 1. Cat code '9' - real serial or sentinel? True cats cluster in month x state.#    claims["CATCD"].value_counts(dropna=False)# 2. Exposure window - rerun fits on 2023-2024 only. ACTYR 125 is immature.# 3. Landlord +0.00733 (p<0.001, $1.6B) suppressed by 'slope > 0 -> flat'.#    Refit without NTR=9. Classic-side item, but the same fallback rule.# 4. Baseline. Confirm the prior release path and that its premium is comparable.# 5. Florida stays in the CDF fit notebook - condo only, and scored output cannot#    answer it.# 6. Note ply_ntr_nbr and the pred_ntr_* arrays run to 49, while#    projected_ntr_*_lr stops at 9. Confirm what happens to the applied LR beyond#    that - the fit caps at ntr 9 (nt6 18), so it may just hold flat.